In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, to_date, count


spark.sql("USE CATALOG pharma_medallion_v2")

DataFrame[]

In [0]:
# Ensure gold schema exists
spark.sql("CREATE SCHEMA IF NOT EXISTS dev_gold")

silver_db = "dev_silver"
gold_db   = "dev_gold"

In [0]:
spark.table(f"{silver_db}.dim_product_sl") \
     .write.mode("overwrite").format("delta") \
     .saveAsTable(f"{gold_db}.dim_product")

In [0]:
spark.table(f"{silver_db}.dim_site_sl") \
     .write.mode("overwrite").format("delta") \
     .saveAsTable(f"{gold_db}.dim_site")

In [0]:
spark.table(f"{silver_db}.dim_equipment_sl") \
     .write.mode("overwrite").format("delta") \
     .saveAsTable(f"{gold_db}.dim_equipment")

In [0]:
bh = spark.table(f"{silver_db}.batch_header_sl")
ev = spark.table(f"{silver_db}.fact_events_src_sl")

fact_batch_quality_events = (
    bh.join(ev.drop('_rescued_data', 'ingest_file_date', 'ingest_ts', 'source_file', 'source_system'), on="batch_id", how="inner")
)

fact_batch_quality_events.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{gold_db}.fact_batch_quality_events")

In [0]:
qc = spark.table(f"{silver_db}.fact_qc_tests_src_sl")

fact_batch_qc_tests = (
    bh.join(qc.drop('_rescued_data','ingest_file_date','ingest_ts', 'source_file','source_system'), on="batch_id", how="inner")
)

fact_batch_qc_tests.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{gold_db}.fact_batch_qc_tests")


In [0]:
kpi_daily_oos = (
    spark.table(f"{gold_db}.fact_batch_quality_events")
        .filter(col("event_type") == "OOS")
        .withColumn("event_date", to_date(col("event_time")))
        .groupBy(
            "event_date",
            "site_id",
            "product_id"
        )
        .agg(
            count("*").alias("oos_count")
        )
)

kpi_daily_oos.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable(f"{gold_db}.kpi_daily_oos")


In [0]:
print("dim_product:", spark.table("dev_gold.dim_product").count())
print("dim_site:", spark.table("dev_gold.dim_site").count())
print("dim_equipment:", spark.table("dev_gold.dim_equipment").count())

print("fact_batch_quality_events:",
      spark.table("dev_gold.fact_batch_quality_events").count())

print("fact_batch_qc_tests:",
      spark.table("dev_gold.fact_batch_qc_tests").count())

print("kpi_daily_oos:",
      spark.table("dev_gold.kpi_daily_oos").count())


dim_product: 3
dim_site: 3
dim_equipment: 5
fact_batch_quality_events: 136
fact_batch_qc_tests: 119
kpi_daily_oos: 0


In [0]:
%sql

select * from dev_gold.fact_batch_quality_events

batch_id,product_id,site_id,equipment_id,process_step,batch_start_time,batch_end_time,batch_status,source_system,ingest_file_date,_rescued_data,ingest_ts,source_file,event_time,event_type,event_value,limit_value,severity,status
BR-202511-00004,P1002,S001,E-STER-01,Compounding,2025-11-18T01:00:00.000Z,2025-11-18T19:00:00.000Z,COMPLETED,MES,2025-11-18,null,2026-01-17T05:36:49.740Z,/Volumes/pharma_medallion_v2/default/raw/batch_header.csv,2025-11-18T03:32:00.000Z,TEMP_EXCURSION_DELTA_C,2.5,2.0,MAJOR,UNDER_INVESTIGATION
BR-202511-00004,P1002,S001,E-STER-01,Compounding,2025-11-18T01:00:00.000Z,2025-11-18T19:00:00.000Z,COMPLETED,MES,2025-11-18,null,2026-01-17T05:36:49.740Z,/Volumes/pharma_medallion_v2/default/raw/batch_header.csv,2025-11-18T10:05:00.000Z,LINE_STOP_MINUTES,5.0,10.0,MINOR,UNDER_INVESTIGATION
BR-202511-00007,P1001,S002,E-PACK-01,FillFinish,2025-11-16T02:00:00.000Z,2025-11-16T14:00:00.000Z,IN_PROGRESS,EBR,2025-11-16,null,2026-01-17T05:36:49.740Z,/Volumes/pharma_medallion_v2/default/raw/batch_header.csv,2025-11-16T06:12:00.000Z,TEMP_EXCURSION_DELTA_C,2.0,2.0,MINOR,CLOSED
BR-202511-00007,P1001,S002,E-PACK-01,FillFinish,2025-11-16T02:00:00.000Z,2025-11-16T14:00:00.000Z,IN_PROGRESS,EBR,2025-11-16,null,2026-01-17T05:36:49.740Z,/Volumes/pharma_medallion_v2/default/raw/batch_header.csv,2025-11-16T09:24:00.000Z,TEMP_EXCURSION_DELTA_C,2.0,2.0,MINOR,UNDER_INVESTIGATION
BR-202511-00007,P1001,S002,E-PACK-01,FillFinish,2025-11-16T02:00:00.000Z,2025-11-16T14:00:00.000Z,IN_PROGRESS,EBR,2025-11-16,null,2026-01-17T05:36:49.740Z,/Volumes/pharma_medallion_v2/default/raw/batch_header.csv,2025-11-16T11:11:00.000Z,LINE_STOP_MINUTES,12.0,10.0,MINOR,CLOSED
BR-202511-00010,P1001,S001,E-FILL-01,Compounding,2025-11-15T13:00:00.000Z,2025-11-15T19:00:00.000Z,IN_PROGRESS,MES,2025-11-15,null,2026-01-17T05:36:49.740Z,/Volumes/pharma_medallion_v2/default/raw/batch_header.csv,2025-11-15T16:22:00.000Z,MISSING_SIGNATURE,1.0,null,MINOR,OPEN
BR-202511-00010,P1001,S001,E-FILL-01,Compounding,2025-11-15T13:00:00.000Z,2025-11-15T19:00:00.000Z,IN_PROGRESS,MES,2025-11-15,null,2026-01-17T05:36:49.740Z,/Volumes/pharma_medallion_v2/default/raw/batch_header.csv,2025-11-15T20:03:00.000Z,MISSING_SIGNATURE,0.0,null,MINOR,UNDER_INVESTIGATION
BR-202511-00010,P1001,S001,E-FILL-01,Compounding,2025-11-15T13:00:00.000Z,2025-11-15T19:00:00.000Z,IN_PROGRESS,MES,2025-11-15,null,2026-01-17T05:36:49.740Z,/Volumes/pharma_medallion_v2/default/raw/batch_header.csv,2025-11-15T22:06:00.000Z,MISSING_SIGNATURE,0.0,null,MINOR,UNDER_INVESTIGATION
BR-202511-00010,P1001,S001,E-FILL-01,Compounding,2025-11-15T13:00:00.000Z,2025-11-15T19:00:00.000Z,IN_PROGRESS,MES,2025-11-15,null,2026-01-17T05:36:49.740Z,/Volumes/pharma_medallion_v2/default/raw/batch_header.csv,2025-11-15T23:26:00.000Z,MISSING_SIGNATURE,0.0,null,MINOR,OPEN
BR-202511-00012,P3001,S001,E-FILL-01,FillFinish,2025-11-22T12:00:00.000Z,2025-11-23T06:00:00.000Z,ON_HOLD,MES,2025-11-22,null,2026-01-17T05:36:49.740Z,/Volumes/pharma_medallion_v2/default/raw/batch_header.csv,2025-11-22T20:37:00.000Z,LINE_STOP_MINUTES,30.0,10.0,MINOR,CLOSED
